# Day 24: FDE Skills 3a and LLM Orchestration
**Influencing, negotiation and conflict management, then LangChain and AI coding agents such as OpenAI Codex**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## Part 1: Negotiation preparation
A simple structure you can reuse before any important conversation.

In [ ]:
prep = {
    "situation": "Sponsor wants 8 features in 2 weeks; team can deliver 5 well",
    "their_position": "All 8 by the board meeting",
    "their_interests": ["Show the board real progress", "Avoid asking for budget twice"],
    "my_interests": ["Quality the team can stand behind", "Keep trust with the sponsor"],
    "options": ["5 finished + demo of 3 planned", "8 with 2 as clearly labelled prototypes", "Board preview of 5, remaining 3 two weeks later"],
    "my_batna": "Deliver 5 on time and present the plan for the rest",
    "ask": "Agree which option by Wednesday",
}
for k, v in prep.items():
    print(f"{k:<16}: {v if isinstance(v, str) else '; '.join(v)}")

## Part 2: LLM orchestration with LangChain
### Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas langchain-core langchain-openai

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas langchain-core langchain-openai
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

In [ ]:
import re, json
import genai_helpers
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableBranch

def offline_model(prompt_value):
    text = prompt_value.to_string()
    if "Return JSON with keys" in text:          # stand-in extractor using regular expressions
        msg = text.split("Message:")[-1]
        amount = re.search(r"([\d,]+(?:\.\d+)?)\s*(INR|JPY|rupees|yen)", msg, re.I)
        name = re.search(r"\b([A-Z][a-z]+)\b", msg)
        date = re.search(r"\d{4}-\d{2}-\d{2}", msg)
        cur = {"rupees": "INR", "yen": "JPY"}.get(amount.group(2).lower(), amount.group(2).upper()) if amount else None
        return json.dumps({"employee": name.group(1) if name else None,
                           "amount": float(amount.group(1).replace(",", "")) if amount else None,
                           "currency": cur, "date": date.group(0) if date else None})
    return genai_helpers.offline_llm(text)

if ONLINE:
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(model=genai_helpers.MODEL, temperature=0, api_key=genai_helpers.OPENAI_API_KEY)
else:
    model = RunnableLambda(offline_model)
print("Model:", "ChatOpenAI" if ONLINE else "offline stand-in")

### A summary chain with batch

In [ ]:
report = ("The new pipeline reduced the nightly batch from 4 hours to 50 minutes. Two data quality checks now stop bad loads. "
          "Infrastructure cost rose by 6 percent due to a larger cluster. The finance team can now close the month two days earlier.")

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You summarise updates for {audience}. Use exactly 3 short bullet points focused on what they care about."),
    ("user", "{text}"),
])
summarise = summary_prompt | model | StrOutputParser()

for audience, out in zip(["a CIO", "a data engineer", "a finance user"],
                         summarise.batch([{"audience": a, "text": report} for a in ["a CIO", "a data engineer", "a finance user"]])):
    print(f"--- {audience} ---\n{out}\n")

### A structured extractor with validation

In [ ]:
from pydantic import BaseModel, ValidationError

class Expense(BaseModel):
    employee: str
    amount: float
    currency: str
    date: str

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the expense. Return JSON with keys employee, amount, currency (INR or JPY), date (YYYY-MM-DD). Return only JSON."),
    ("user", "Message: {message}"),
])
def parse_json(text):
    return json.loads(text[text.index("{"): text.rindex("}") + 1])

extract = extract_prompt | model | StrOutputParser() | RunnableLambda(parse_json)

messages = ["Mei spent 4,200 JPY on a client lunch on 2026-11-20.",
            "Taxi to the airport for Arjun, 1,350 INR, 2026-11-21.",
            "Hotel in Osaka: Kenji paid 14,800 yen on 2026-11-22.",
            "Ravi bought a keyboard for 2,999 rupees on 2026-11-23.",
            "Team dinner, no amount recorded yet."]
valid = 0
for m in messages:
    try:
        e = Expense(**extract.invoke({"message": m}))
        valid += 1
        print("OK  ", e)
    except (ValidationError, ValueError, TypeError) as err:
        print("FAIL", m, "->", type(err).__name__)
print(f"Valid: {valid}/{len(messages)}  (the last message is meant to fail)")
print("Look closely: valid does not mean correct. Pydantic checks the shape of the data, not the truth.",
      "Offline, the regex stand-in picks 'Taxi' and 'Hotel' as names. Check samples by eye and add tests for meaning too.")

### A RAG chain

In [ ]:
%%writefile handbook.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

In [ ]:
import numpy as np
text = open("handbook.md").read()
docs = [s.strip() for s in re.split(r"(?<=[.!?])\s+", re.sub(r"#+ .*\n", "", text)) if s.strip()]
doc_vecs = embed(docs)

def search(q, k=3):
    idx = np.argsort(doc_vecs @ embed(q)[0])[::-1][:k]
    return "\n".join(f"[d{i}] {docs[i]}" for i in idx)

grounded = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context. Cite ids like [d3]. If it is not there, say \"I don't know\"."),
    ("user", "Context:\n{context}\n\nQuestion: {question}"),
])
rag = {"context": RunnableLambda(search), "question": RunnablePassthrough()} | grounded | model | StrOutputParser()
print(rag.invoke("When does carried forward leave expire?"))

### Routing with RunnableBranch

In [ ]:
chat = ChatPromptTemplate.from_messages([("system", "You are a friendly assistant. Reply in one sentence."), ("user", "{question}")]) | model | StrOutputParser()
HR_WORDS = ("leave", "expense", "remote", "laptop", "security", "hours", "sick")
is_hr = lambda x: any(w in x["question"].lower() for w in HR_WORDS)

router = RunnableBranch(
    (is_hr, RunnableLambda(lambda x: x["question"]) | rag),
    RunnableLambda(lambda x: {"question": x["question"]}) | chat,
)
for q in ["How many sick days do I get?", "Hello, how are you today?"]:
    print(f"{q:<32} -> {'rag ' if is_hr({'question': q}) else 'chat'} -> {router.invoke({'question': q})}")

### TODO 1: Add a third route for expense questions that uses the extractor when the message contains an amount.

In [ ]:
# TODO: write your code here

## Part 3: Briefing a coding agent
Coding agents such as OpenAI Codex read `AGENTS.md` for project instructions. Write it like onboarding notes for a new colleague.

In [ ]:
%%writefile AGENTS.md
# AGENTS.md: Task Board

## Setup
- Python 3.11. Install: pip install -r requirements.txt -r requirements-dev.txt
- Front end: cd web && npm ci

## Commands
- Tests (must pass before proposing changes): pytest -q && (cd web && npm test)
- Lint: ruff check .
- Run API: uvicorn app.main:app --reload

## Conventions
- FastAPI routers live in app/routes/, one file per resource. Pydantic models in app/schemas.py.
- Every new endpoint needs tests in tests/ covering success and error cases.
- Keep functions small; type hints on all public functions.

## Do not
- Do not modify infra/ or azure-pipelines.yml without being asked.
- Never commit secrets, .env files or customer data.

In [ ]:
%%writefile TASK_priority_filter.md
# Task: add a priority filter to GET /api/tasks

## Goal
Users want to see only high priority tasks.

## Details
- Add optional query parameter `priority` (low | medium | high) to GET /api/tasks in app/routes/tasks.py.
- Invalid values return 422 (use an Enum).
- In web/src/components/Filters.jsx add a priority dropdown that calls the API with the parameter.

## Acceptance criteria
- GET /api/tasks?priority=high returns only high priority tasks.
- Existing tests still pass; add at least 3 new API tests and 1 UI test.

## Out of scope
- Sorting, pagination, database migrations.

In [ ]:
for f in ["AGENTS.md", "TASK_priority_filter.md"]:
    body = open(f).read()
    print(f"{f}: {len(body.splitlines())} lines, sections: {[l for l in body.splitlines() if l.startswith('## ')]}")

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)